# 🍳 ChefMind — Fine-Tuning LLaMA 3.1 8B
### AI Chef that generates recipes from ingredients using LLaMA 3.1 8B + LoRA

## 🔥 1️⃣ Check GPU

In [1]:
import torch
import warnings
warnings.filterwarnings("ignore")

print("✅ Libraries imported successfully!")
print(f"🔥 GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🖥️  GPU Name: {torch.cuda.get_device_name(0)}")

✅ Libraries imported successfully!
🔥 GPU Available: True
🖥️  GPU Name: Tesla T4


## 📦 2️⃣ Install Libraries

In [2]:
%%capture
!pip install transformers==4.52.4 peft accelerate bitsandbytes trl datasets

## 🤖 3️⃣ Load Base Model

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "unsloth/Meta-Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map          = "auto",
)

print("✅ Model loaded successfully!")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Model loaded successfully!


## 🎯 4️⃣ Apply LoRA

In [4]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r             = 16,
    lora_alpha    = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_dropout  = 0.1,
    bias          = "none",
    task_type     = "CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

def print_trainable_parameters(model):
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param:.2f}%")

print_trainable_parameters(model)
print("✅ LoRA applied successfully!")

trainable params: 41943040 || all params: 4582543360 || trainable%: 0.92%
✅ LoRA applied successfully!


## 📊 5️⃣ Load Dataset

In [5]:
from datasets import load_dataset

raw_dataset = load_dataset("tengomucho/simple_recipes")

print(f"✅ Dataset loaded successfully!")
print(f"📊 Number of recipes: {len(raw_dataset['train'])}")
print(f"📋 Columns: {raw_dataset['train'].column_names}")
print(f"\n🍽️  Sample Recipe:")
print(f"Name: {raw_dataset['train'][0]['names']}")
print(f"Recipe: {raw_dataset['train'][0]['recipes'][:300]}...")

✅ Dataset loaded successfully!
📊 Number of recipes: 20000
📋 Columns: ['recipes', 'names']

🍽️  Sample Recipe:
Name: Arriba  baked winter squash mexican style
Recipe: - Make a choice and proceed with recipe
- Depending on size of squash, cut into half or fourths
- Remove seeds
- For spicy squash, drizzle olive oil or melted butter over each cut squash piece
- Season with mexican seasoning mix ii
- For sweet squash, drizzle melted honey, butter, grated piloncillo ...


## 📝 6️⃣ Formatting Function

In [6]:
def formatting_prompts_func(examples):
    names    = examples["names"]
    recipes  = examples["recipes"]
    texts    = []

    for name, recipe in zip(names, recipes):
        text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are ChefMind, a creative AI chef. When given a dish name or ingredients, you provide a complete, delicious recipe with clear steps.<|eot_id|><|start_header_id|>user<|end_header_id|>

Give me a recipe for: {name}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{recipe}<|eot_id|>"""
        texts.append(text)

    return {"text": texts,}

pass

## 🗺️ 7️⃣ Map Dataset

In [7]:
dataset = raw_dataset["train"].map(
    formatting_prompts_func,
    batched = True,
)

print(f"✅ Dataset formatted successfully!")
print(f"📊 Total examples: {len(dataset)}")
print(f"\n📝 Sample formatted text (first 500 chars):")
print(dataset[0]["text"][:500])

✅ Dataset formatted successfully!
📊 Total examples: 20000

📝 Sample formatted text (first 500 chars):
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are ChefMind, a creative AI chef. When given a dish name or ingredients, you provide a complete, delicious recipe with clear steps.<|eot_id|><|start_header_id|>user<|end_header_id|>

Give me a recipe for: Arriba  baked winter squash mexican style<|eot_id|><|start_header_id|>assistant<|end_header_id|>

- Make a choice and proceed with recipe
- Depending on size of squash, cut into half or fourths
- Remove seeds
- For spicy squash, d


## 🏋️ 8️⃣ Configure SFTTrainer

In [8]:
from trl import SFTTrainer, SFTConfig
import torch

trainer = SFTTrainer(
    model         = model,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        max_seq_length              = 512,
        warmup_steps                = 10,
        max_steps                   = 100,
        learning_rate               = 2e-4,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        optim                       = "paged_adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "linear",
        dataset_text_field          = "text",
        dataset_num_proc            = 2,
        packing                     = False,
        logging_steps               = 1,
        report_to                   = "none",
        seed                        = 3407,
        output_dir                  = "outputs",
    ),
)

print("✅ Trainer configured!")

average_tokens_across_devices is set to True but it is invalid when world size is1. Turn it to False automatically.
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


✅ Trainer configured!


## 🚀 9️⃣ Run Training

In [9]:
trainer_stats = trainer.train()

print(f"✅ Training completed!")
print(f"📉 Final Loss:         {trainer_stats.training_loss:.4f}")
print(f"⏱️  Training Time:      {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f"🔄 Total Steps:        {trainer_stats.metrics['train_steps_per_second']:.2f} steps/sec")

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
1,2.875900
2,2.896600
3,2.780500
4,2.430100
5,2.021100
6,1.955400
7,1.774900
8,1.573900
9,1.439900
10,1.379400


Step,Training Loss
1,2.875900
2,2.896600
3,2.780500
4,2.430100
5,2.021100
6,1.955400
7,1.774900
8,1.573900
9,1.439900
10,1.379400


✅ Training completed!
📉 Final Loss:         1.3002
⏱️  Training Time:      8612.63 seconds
🔄 Total Steps:        0.01 steps/sec


## 🍳 1️⃣0️⃣ Test Inference

In [10]:
from transformers import TextStreamer
from peft import PeftModel

model.eval()

messages = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are ChefMind, a creative AI chef. When given a dish name or ingredients, you provide a complete, delicious recipe with clear steps.<|eot_id|><|start_header_id|>user<|end_header_id|>

Give me a recipe for: Garlic Butter Pasta<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

inputs = tokenizer(messages, return_tensors="pt").to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print("🍳 ChefMind is cooking up a recipe...\n")
print("=" * 50)

_ = model.generate(
    **inputs,
    streamer       = text_streamer,
    max_new_tokens = 512,
    use_cache      = True,
)

🍳 ChefMind is cooking up a recipe...

- Bring a large pot of salted water to a boil
- Cook pasta for 8 minutes, or until al dente
- Drain, reserving 1 / 2 cup pasta water
- In a large skillet, melt butter over medium heat
- Add garlic and cook for 1 minute, or until fragrant
- Add pasta and reserved pasta water
- Toss to coat
- Season with salt and pepper to taste<|eot_id|>


## 💾 1️⃣1️⃣ Save Model

In [11]:
model.save_pretrained("chefmind-lora")
tokenizer.save_pretrained("chefmind-lora")

print("✅ Model saved successfully!")
print("📁 Saved to: chefmind-lora/")

✅ Model saved successfully!
📁 Saved to: chefmind-lora/


## ☁️ 1️⃣2️⃣ Save to Google Drive

In [13]:
from google.colab import drive
import shutil

# Force remount in case it's stuck
drive.mount("/content/drive", force_remount=True)

shutil.copytree("chefmind-lora", "/content/drive/MyDrive/ChefMind/chefmind-lora", dirs_exist_ok=True)
print("✅ Model saved to Google Drive!")

Mounted at /content/drive
✅ Model saved to Google Drive!


In [18]:
from huggingface_hub import login, create_repo

HF_TOKEN = "YOUR_HF_TOKEN"
HF_REPO  = "Osamaelbably87/chefmind-lora"   # ← الاسم الصح

login(token=HF_TOKEN)

create_repo(HF_REPO, repo_type="model", exist_ok=True, token=HF_TOKEN)
print(f"✅ Repo created!")


model.push_to_hub(HF_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)

print(f"✅ Done! 🔗 https://huggingface.co/{HF_REPO}")

✅ Repo created!


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  557kB /  168MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  chefmind-lora/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

✅ Done! 🔗 https://huggingface.co/Osamaelbably87/chefmind-lora
